# Education 360° — Data Quality Validation
**Objective:** Check missing values, duplicates, candidate keys, date ranges, and referential integrity. This notebook reports issues; it does not silently modify source data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["branches_dim.csv", "braches_dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses_dim.csv"],
    "date table": ["DateTable.csv", "date_table.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students_dim.csv"]
}

def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

## 1. Missing values and duplicate rows

In [ ]:
quality_rows = []
for name, df in datasets.items():
    quality_rows.append({
        "Table": name,
        "Rows": len(df),
        "Missing Cells": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    })
display(pd.DataFrame(quality_rows))

## 2. Candidate primary-key checks

In [ ]:
primary_keys = {
    "admissions": "admission id",
    "branches dim": "branch id",
    "calls": "call id",
    "courses dim": "course_id",
    "date table": "Date",
    "employees": "emp id",
    "fees": "payment id",
    "leads": "lead id",
    "marketing": "campaign id",
    "placements": "placement id",
    "students dim": "student id"
}
key_results = []
for table, key in primary_keys.items():
    df = datasets.get(table)
    if df is None or key not in df.columns:
        key_results.append({"Table":table,"Key":key,"Status":"TABLE/COLUMN NOT FOUND"})
        continue
    key_results.append({
        "Table":table, "Key":key, "Status":"Checked",
        "Missing Keys":int(df[key].isna().sum()),
        "Duplicate Keys":int(df[key].duplicated().sum()),
        "Unique Keys":int(df[key].nunique(dropna=True))
    })
display(pd.DataFrame(key_results))

## 3. Date-range checks

In [ ]:
date_columns = {
    "admissions": ["admission date"],
    "calls": ["call date"],
    "date table": ["Date"],
    "leads": ["generated date"],
    "students dim": ["admission_date"]
}
for table, cols in date_columns.items():
    df = datasets.get(table)
    if df is None: 
        continue
    for col in cols:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors="coerce")
            print(table, col, "| invalid:", int(parsed.isna().sum()),
                  "| min:", parsed.min(), "| max:", parsed.max())

## 4. Foreign-key / orphan checks

In [ ]:
relationships = [
    ("admissions", "student id", "students dim", "student id"),
    ("admissions", "branch id", "branches dim", "branch id"),
    ("admissions", "course id", "courses dim", "course_id"),
    ("admissions", "counselor id", "employees", "emp id"),
    ("calls", "student id", "students dim", "student id"),
    ("leads", "branch id", "branches dim", "branch id"),
    ("fees", "student id", "students dim", "student id"),
    ("placements", "student id", "students dim", "student id"),
    ("students dim", "branch id", "branches dim", "branch id"),
    ("students dim", "course id", "courses dim", "course_id")
]
orphan_results = []
for child, child_col, parent, parent_col in relationships:
    a, b = datasets.get(child), datasets.get(parent)
    if a is None or b is None or child_col not in a or parent_col not in b:
        orphan_results.append({"Relationship":f"{child}.{child_col} -> {parent}.{parent_col}",
                               "Status":"TABLE/COLUMN NOT FOUND"})
        continue
    parent_values = set(b[parent_col].dropna())
    orphans = a.loc[a[child_col].notna() & ~a[child_col].isin(parent_values), child_col]
    orphan_results.append({
        "Relationship":f"{child}.{child_col} -> {parent}.{parent_col}",
        "Orphan Rows":len(orphans),
        "Distinct Orphan Values":orphans.nunique(),
        "Status":"Checked"
    })
display(pd.DataFrame(orphan_results))

## 5. Standardization review

In [ ]:
for table, df in datasets.items():
    suspicious = [c for c in df.columns if c != c.strip()]
    if suspicious:
        print(table, "column names with leading/trailing spaces:", suspicious)

## Conclusion
Record confirmed issues separately from expected differences in table grain. Do not join Leads to Students by name alone; the source has no shared unique student key.